# Development Challenges and Engineering Decisions

This document summarizes notable technical challenges encountered while developing LotStack, including their causes, solutions, and resulting engineering decisions.

## 1. API Request Payload Formatting

### Problem

An external API request produced a `too many values to unpack` error because the request payload used incorrect Python syntax:

```python
{"format:json"}
```

This created a set containing one string instead of a dictionary containing a key-value pair.

### Resolution

The request payload was corrected to:

```python
{"format": "json"}
```

Additional response validation was added to make API errors easier to identify.

### Lesson

External API payloads should be validated against the expected request structure before being sent.

---

## 2. Data Collection Architecture

### Problem

Collecting vehicle data directly from auction websites proved unreliable because some platforms use client-side JavaScript, bot-detection systems, authentication, and frequently changing page structures.

Live auction information can also change rapidly, making one-time HTML requests insufficient for some use cases.

### Decision

LotStack was reorganized around an ingestion pipeline in which scraping is one possible data source.

```text
Auction Source
      ↓
Source-Specific Collector
      ↓
Raw Listing Data
      ↓
Validation and Normalization
      ↓
VIN Enrichment
      ↓
PostgreSQL
      ↓
Analysis and Ranking API
```

The pipeline prioritizes structured data sources when available. Site-specific scraping can still be used for publicly available information when a structured source is unavailable.

### Lesson

Scraping and ingestion are not competing approaches. Scraping can be one component of an ingestion pipeline that also handles validation, transformation, normalization, enrichment, storage, and analysis.

---

## 3. Large Entity Constructors

### Problem

The vehicle entity accumulated too many constructor parameters as additional vehicle, auction, expense, and analysis fields were introduced.

Large constructors made object creation difficult to read and increased the possibility of passing values in the wrong order.

### Resolution

Responsibilities were divided across entities and DTOs. Related data was separated into vehicle, expense, history, transportation, and analysis models.

### Lesson

A large constructor often indicates that a class has too many responsibilities. Database entities, request DTOs, response DTOs, and domain services should have clearly separated purposes.

---

## 4. Primitive and Wrapper Type Inconsistency

### Problem

Some entity fields were declared using primitive types such as `int`, while their constructors, getters, and setters used wrapper types such as `Integer`.

This created inconsistent null-handling behavior.

### Resolution

Types were standardized according to whether a value could legitimately be absent.

```java
private Integer mileage;
```

Wrapper types are used when database values may be `NULL`. Primitive types are used when a value must always exist.

### Lesson

Entity field types should match their accessor types and database nullability requirements.

---

## 5. Database Naming Conventions

### Problem

Java camelCase names and PostgreSQL snake_case names were used inconsistently, making entity-to-column mappings harder to understand.

### Resolution

The project adopted the following conventions:

- `camelCase` for Java fields
- `snake_case` for PostgreSQL tables and columns
- Explicit JPA mappings where required

```java
@Column(name = "purchase_price")
private BigDecimal purchasePrice;
```

### Lesson

Application and database naming conventions can differ, but their mappings must remain explicit and consistent.

---

## 6. JPA Entities and Java Records

### Problem

Java records were considered for database entities because of their concise syntax. However, records are immutable and final, while JPA entities commonly require mutable fields, a no-argument constructor, and support for Hibernate proxies.

### Decision

Database entities remain standard Java classes. Records may still be used for immutable request and response DTOs.

```java
@Entity
public class VehicleEntity {

    protected VehicleEntity() {
    }

    // Entity fields and methods
}
```

### Lesson

Java records are useful for immutable data transfer, but standard classes are generally more appropriate for JPA-managed entities.

---

## 7. Spring Data JPA and CRUD Operations

### Question

Should the application implement custom CRUD operations or use Spring Data JPA?

### Decision

Spring Data JPA repositories provide the standard CRUD operations required by the application.

```java
public interface VehicleRepository
        extends JpaRepository<VehicleEntity, Long> {
}
```

Custom repository methods are added only when the application requires queries beyond the inherited CRUD operations.

### Lesson

CRUD describes the operations being performed, while JPA is the persistence technology used to perform them. They are not mutually exclusive choices.

---

## 8. Missing Spring Annotations

### Problem

Some classes were not detected by Spring because they were missing annotations required for component scanning and dependency injection.

Examples include:

```java
@Service
@Repository
@RestController
@Entity
```

### Resolution

Classes were reviewed to ensure each one used the annotation appropriate for its responsibility and remained within the application’s component-scanning hierarchy.

### Lesson

Spring annotations determine how classes participate in the application container. Missing or incorrect annotations can prevent dependency injection even when the underlying Java code compiles successfully.

---

## 9. Repository Method Type Signatures

### Problem

Repository methods were called with argument or return types that did not match their declared signatures.

This occurred when entity identifiers, DTOs, and entity objects were used inconsistently across the controller, service, and repository layers.

### Resolution

Method signatures were standardized across the application layers.

```text
Controller
    ↓ Request and response DTOs
Service
    ↓ Business logic and entity operations
Repository
    ↓ Entity types and identifiers
Database
```

Repositories operate primarily on entity types and their identifiers. Conversion between DTOs and entities occurs in the service layer.

### Lesson

Consistent type boundaries make application layers easier to understand, test, and maintain.

---

## 10. Financial Calculation Consistency

### Problem

Vehicle totals appeared inconsistent because the dashboard displayed repair costs separately, while total investment also included auction fees, transportation, paperwork, and other expenses.

### Resolution

The financial definitions were standardized:

```text
Repair Cost = Sum of REPAIR expenses

Other Costs =
    AUCTION FEE
  + PAPERWORK
  + TRANSPORT
  + OTHER

Total Invested =
    Purchase Price
  + Repair Cost
  + Other Costs

Profit =
    Selling Price
  - Total Invested

ROI =
    Profit / Total Invested × 100
```

### Planned Improvement

Financial totals will be calculated directly from the underlying expense records and verified through automated tests instead of being maintained as separate manually entered values.

### Lesson

Derived financial values should use one source of truth. Maintaining the same total in multiple places increases the possibility of inconsistent data.

---

## 11. Rule-Based Ranking Before Machine Learning

### Challenge

The planned machine-learning model requires enough clean historical data for training and evaluation. Implementing an untested model before establishing a working baseline would make its performance difficult to measure.

### Decision

LotStack currently uses a transparent rule-based ranking system based on purchase price, repair costs, resale potential, title condition, and risk.

This system provides an operational baseline that can later be compared with the machine-learning model.

### Planned Evaluation

The machine-learning model will be evaluated using:

- Resale-price prediction error
- Expected versus realized profit
- Expected versus realized ROI
- Ranking accuracy
- Performance compared with the rule-based baseline

### Lesson

A simpler working baseline makes it possible to determine whether a more complex model actually improves the system.

---

## Current Development Priorities

- Complete and evaluate the machine-learning ranking model
- Add automated validation for vehicle expenses and portfolio totals
- Expand supported auction-data integrations
- Improve scheduling and notification workflows
- Compare predictions with actual resale outcomes
- Add explainable ranking recommendations